In [ ]:
import pandas as pd
from tqdm.notebook import tqdm
from sklearn.model_selection import train_test_split
import numpy as np
import pickle

DATA = "umls"

# df = pd.read_csv(f"/home/cc/cc_my_mounting_point/kg/{DATA}/kg.csv", sep=",", low_memory=False)
# save_path = f"/home/cc/phd/KGEmbeddings/data/{DATA}/"

## PrimeKG combo
# combinations = [
#     ("drug_protein", "disease_protein", "pathway_protein"),        # 9.0
#     ("drug_protein", "exposure_protein", "bioprocess_protein"),    # 8.0
#     ("anatomy_protein_present", "disease_protein", "molfunc_protein"),  # 8.0
#     ("disease_phenotype_positive", "phenotype_protein", "drug_effect"), # 7.5
#     ("exposure_disease", "disease_disease", "indication"),         # 7.5
#     ("drug_drug", "drug_protein", "contraindication"),             # 7.0
#     ("disease_protein", "pathway_protein", "drug_effect"),         # 7.0
#     ("phenotype_protein", "disease_protein", "pathway_protein"),   # 6.5
#     ("indication", "off-label use", "drug_effect"),                # 6.0
#     ("anatomy_protein_absent", "anatomy_protein_present", "cellcomp_protein"), # 6.0
#     ("drug_effect", "drug_drug", "indication"),                    # 6.0
# ]

## small set umls
# combinations = [
#     ("may_treat", "contraindicated_with_disease", "manifestation_of"),
#     ("may_treat", "has_phenotype", "has_finding_site"),
#     ("causative_agent_of", "manifestation_of", "associated_morphology_of"),
#     ("used_for", "therapeutic_class_of", "physiologic_effect_of"),
#     ("has_active_ingredient", "has_ingredient", "mechanism_of_action_of"),
#     ("has_finding_site", "location_of", "has_component"),
#     ("clinically_associated_with", "associated_with", "co-occurs_with"),
#     ("pathological_process_of", "manifestation_of", "has_phenotype"),
# ]

# Full set umls
combinations = [
    ("may_treat", "contraindicated_with_disease", "manifestation_of"),
    ("may_treat", "has_phenotype", "has_finding_site"),
    ("causative_agent_of", "manifestation_of", "associated_morphology_of"),
    ("used_for", "therapeutic_class_of", "physiologic_effect_of"),
    ("has_active_ingredient", "has_ingredient", "mechanism_of_action_of"),
    ("has_finding_site", "location_of", "has_component"),
    ("associated_morphology_of", "pathological_process_of", "has_finding_site"),
    ("clinically_associated_with", "associated_with", "co-occurs_with"),
    ("pathological_process_of", "manifestation_of", "has_phenotype"),
    ("has_procedure_site", "location_of", "method_of"),
    ("has_direct_procedure_site", "location_of", "method_of"),
    ("has_component", "has_focus", "has_phenotype"),
    ("associated_with", "has_finding_site", "has_phenotype"),
    ("may_treat", "mechanism_of_action_of", "has_phenotype"),
    ("associated_with", "has_phenotype", "manifestation_of"),
]


rel_set = {r for combo in combinations for r in combo}

In [ ]:
def compute_shared_tails(df_filtered, combinations, number_of_proj):
    # Pre-compute heads per tail per relation
    # Map: (relation, tail) → set of heads
    rel_tail_to_heads = (
        df_filtered
        .groupby(['relation_id', 'tail_id'])['head_id']
        .agg(lambda s: set(s))
        .reset_index()
    )
    # Pivot or create a dict for quick lookup
    # relation → { tail → heads_set }
    rel_to_tail_heads = {}
    for rel in set(r for combo in combinations for r in combo):
        sub = rel_tail_to_heads[rel_tail_to_heads['relation_id'] == rel]
        rel_to_tail_heads[rel] = dict(zip(sub['tail_id'], sub['head_id']))

    queries = []
    for (r1, r2, r_final) in tqdm(combinations , desc="Processing combinations"):
        print(f"Processing combination: {r1}, {r2} -> {r_final}")
        # Find tails that appear in both r1 and r2
        tails_r1 = set(rel_to_tail_heads.get(r1, {}).keys())
        tails_r2 = set(rel_to_tail_heads.get(r2, {}).keys())
        shared_tails = tails_r1.intersection(tails_r2)
        
        for tail in shared_tails:
            heads1 = list(rel_to_tail_heads[r1][tail])[:number_of_proj]
            heads2 = list(rel_to_tail_heads[r2][tail])[:number_of_proj]
            
            if not heads1 or not heads2:
                continue
            
            # Now find results for r_final: tail → results
            heads_for_final = rel_to_tail_heads.get(r_final, {}).get(tail, set())
            if not heads_for_final:
                continue
            
            # Build queries for cross-product of heads1 × heads2
            for h1 in heads1:
                for h2 in heads2:
                    queries.append({
                        "h1": h1,
                        "h2": h2,
                        "tail": tail,
                        "r1": r1,
                        "r2": r2,
                        "r_final": r_final,
                        "res": list(heads_for_final)
                    })

    np.random.shuffle(queries)
    return queries

In [ ]:
import json

number_of_proj = 2

# unique_nodes = pd.Index(sorted(set(df_filtered["x_index"]).union(set(df_filtered["y_index"]))))
# node2id = pd.Series(data=range(len(unique_nodes)), index=unique_nodes)

# df_filtered['x_index'] = df_filtered['x_index'].map(node2id)
# df_filtered['y_index'] = df_filtered['y_index'].map(node2id)

train = pd.read_csv(f"/home/cc/phd/KGEmbeddings/data/{DATA}/train.csv")
test = pd.read_csv(f"/home/cc/phd/KGEmbeddings/data/{DATA}/test.csv")
val = pd.read_csv(f"/home/cc/phd/KGEmbeddings/data/{DATA}/valid.csv")

df_filtered = pd.concat([train, test, val], ignore_index=True)
# df_filtered = df_filtered[df_filtered["relation"].isin(rel_set)]

with open(f"/home/cc/phd/KGEmbeddings/data/{DATA}/rel_map.json") as f:
    relation_to_id = json.load(f)

# relation_to_id = pd.Series(data=rel_map["relation_id"].values, index=rel_map["relation"].values)

# rel_map = pd.read_csv(f"/home/cc/phd/KGEmbeddings/data/{DATA}/relation_map.csv")
# relation_to_id = pd.Series(data=rel_map["relation_id"].values, index=rel_map["relation"].values)

# relation_to_id = relation_to_id.to_dict()
# relation_to_id

id_combinations = [
    tuple(relation_to_id[name] for name in combo)
    for combo in combinations
]

queries = compute_shared_tails(df_filtered, id_combinations, number_of_proj)

with open(f'/home/cc/phd/KGEmbeddings/queries/{DATA}/queries2.pkl', 'wb') as f:
    pickle.dump(queries, f)

len(queries)